# Marketstack — Stock & Market Data

Demonstrates **ticker search**, **ticker info**, **EOD data**, **intraday prices**,
**real-time quotes**, **exchanges**, **indices**, **currencies**, and **timezones** tools
using the [Marketstack API](https://marketstack.com).

Requires a `MARKETSTACK_ACCESS_KEY` environment variable set in `.env`.

---
## Setup

In [ ]:
!pip install -q -e "../.."
!pip install -q -r ../requirements.txt

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from demos.shared.llm_setup import get_llm
from pymcpx.marketstack import (
    MarketstackCurrenciesTool,
    MarketstackEodTool,
    MarketstackExchangesTool,
    MarketstackIndexListTool,
    MarketstackIntradayTool,
    MarketstackListTickersTool,
    MarketstackRealtimePriceTool,
    MarketstackTickerInfoTool,
    MarketstackTimezonesTool,
)

llm = get_llm()

tools = [
    MarketstackListTickersTool(),
    MarketstackTickerInfoTool(),
    MarketstackEodTool(),
    MarketstackIntradayTool(),
    MarketstackRealtimePriceTool(),
    MarketstackExchangesTool(),
    MarketstackIndexListTool(),
    MarketstackCurrenciesTool(),
    MarketstackTimezonesTool(),
]

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a financial data assistant. Use the marketstack tools to answer "
            "questions about stocks and markets.\n\n"
            "Available tools:\n"
            "- marketstack_list_tickers(search, exchange, limit, offset) \u2014 search and list stock tickers\n"
            "- marketstack_ticker_info(symbol, exchange, use_tickerinfo_endpoint, limit, offset) \u2014 ticker details (sector, industry, employees)\n"
            "- marketstack_eod(symbols, date, latest, exchange, date_from, date_to, sort, limit, offset) \u2014 End-of-Day stock data for one or multiple tickers\n"
            "- marketstack_intraday(symbols, interval, date, latest, exchange, date_from, date_to, sort, limit, offset, after_hours) \u2014 intraday stock prices at intervals from 1min to 1hour\n"
            "- marketstack_realtime_price(ticker, exchange) \u2014 real-time stock price for a single symbol\n"
            "- marketstack_exchanges(mic, endpoint_type, symbols, date, interval, date_from, date_to, sort, limit, offset, search) \u2014 list/query stock exchanges\n"
            "- marketstack_indices(index, limit, offset) \u2014 list indices or get details for a specific index\n"
            "- marketstack_currencies(limit, offset) \u2014 list supported currencies\n"
            "- marketstack_timezones(limit, offset) \u2014 list supported timezones",
        ),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
print("\u2705 Agent ready. Running scenarios...")

In [ ]:
result = executor.invoke(
    {"input": "Search for tickers related to 'Apple' and show me details about AAPL."}
)
print(result["output"])

In [ ]:
result = executor.invoke({"input": "Get the latest EOD data for AAPL only."})
print(result["output"])

In [ ]:
result = executor.invoke(
    {"input": "List the stock exchanges for NASDAQ, NYSE, and London Stock Exchange (limit 3)."}
)
print(result["output"])

In [ ]:
result = executor.invoke(
    {"input": "Get the real-time price for AAPL. Also list the available currencies."}
)
print(result["output"])

In [ ]:
result = executor.invoke(
    {"input": "List the stock market indices (limit 5) and show me which timezones are supported."}
)
print(result["output"])